# TermoRed S.A. — Capa Bronze

Objetivo: ingerir las 5 fuentes desde el Data Lake y persistirlas en
Parquet sin alterar su contenido.

Criterio de diseño: todo se lee como `string`.
Bronze solo agrega metadatos de linaje (`fecha_carga`, `archivo_origen`).

Requiere un cluster clásico (all-purpose). Ya que con cómputo *serverless* no se
pueden fijar las claves de storage por `spark.conf`.

## 0. Configuración de acceso

Lo correcto es usar un secret scope en lugar de la clave en
texto plano:

```bash
databricks configure --token
# Host: https://adb-XXXXXXXXXXXX.XX.azuredatabricks.net
# Token: dapi...

databricks secrets create-scope termored_scope
databricks secrets put-secret termored_scope storage_key --string-value "CLAVE"
databricks secrets list-secrets termored_scope
```

Luego: `clave = dbutils.secrets.get("termored_scope", "storage_key")`

## 1. Conexión y rutas

In [0]:
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType

CUENTA     = "destorageintegration"
CONTENEDOR = "termored-central-us-dev"

CLAVE = dbutils.secrets.get("termored_scope", "storage_key")

spark.conf.set(f"fs.azure.account.key.{CUENTA}.dfs.core.windows.net", CLAVE)

BASE   = f"abfss://{CONTENEDOR}@{CUENTA}.dfs.core.windows.net"
RAW    = f"{BASE}/0_raw"
BRONZE = f"{BASE}/1_bronze"
SILVER = f"{BASE}/2_silver"
GOLD   = f"{BASE}/3_gold"

display(dbutils.fs.ls(RAW))

path,name,size,modificationTime
abfss://termored-central-us-dev@destorageintegration.dfs.core.windows.net/0_raw/bancos.csv,bancos.csv,1778,1786494151000
abfss://termored-central-us-dev@destorageintegration.dfs.core.windows.net/0_raw/cortes_energia.json,cortes_energia.json,5353,1786568913000
abfss://termored-central-us-dev@destorageintegration.dfs.core.windows.net/0_raw/lecturas.csv,lecturas.csv,111430,1786494152000
abfss://termored-central-us-dev@destorageintegration.dfs.core.windows.net/0_raw/personal.csv,personal.csv,4268,1786494151000
abfss://termored-central-us-dev@destorageintegration.dfs.core.windows.net/0_raw/sensores.csv,sensores.csv,8210,1786494151000


## 2. Función de guardado en Parquet

`coalesce(1)` fuerza un único archivo de salida. Spark lo escribe con un
nombre autogenerado (`part-00000-....parquet`) dentro de una carpeta, así que
se escribe a una carpeta temporal, se renombra el archivo y se limpia el
temporal. El sufijo de fecha permite conservar el histórico de cargas.

In [0]:
def guardar_parquet(df, destino, nombre, versionar=False):
    """Escribe df como un único .parquet dentro de `destino`."""
    temp_path = f"{destino}_tmp"

    (df.coalesce(1)
       .write
       .mode("overwrite")
       .option("compression", "snappy")
       .parquet(temp_path))

    archivo = [f.path for f in dbutils.fs.ls(temp_path)
               if f.name.endswith(".parquet")][0]

    if versionar:
        sello = datetime.now().strftime("%Y%m%d_%H%M%S")
        nombre_final = f"{destino}/{nombre}_{sello}.parquet"
    else:
        nombre_final = f"{destino}/{nombre}.parquet"

    dbutils.fs.mv(archivo, nombre_final)
    dbutils.fs.rm(temp_path, recurse=True)
    return nombre_final

## 3. Lectura de los CSV

Se usa `inferSchema=False` porque fuerza todas las columnas a `string`.
Si Spark infiriera tipos, convertiría a `null` (o descartaría) los valores
defectuosos como `-999`, comas decimales, fechas en `DD/MM/YYYY` y se perdería
la evidencia que hay que reportar.

`mode=PERMISSIVE` evita que una fila mal formada aborte la lectura completa.

In [0]:
OPCIONES_CSV = {
    "header":      "true",
    "inferSchema": "false",
    "encoding":    "UTF-8",
    "quote":       '"',
    "escape":      '"',
    "multiLine":   "false",
    "mode":        "PERMISSIVE",
}

ARCHIVOS_CSV = ["bancos", "sensores", "personal", "lecturas"]

dfs = {}

for nombre in ARCHIVOS_CSV:
    ruta = f"{RAW}/{nombre}.csv"
    df = spark.read.options(**OPCIONES_CSV).csv(ruta)

    # Metadatos de linaje: qué se cargó, desde dónde y cuándo
    df = (df
          .withColumn("archivo_origen", F.lit(ruta))
          .withColumn("fecha_carga",    F.current_timestamp()))

    dfs[nombre] = df
    df.createOrReplaceTempView(f"bronze_{nombre}")
    print(f"{nombre:12s} -> {df.count():6d} filas | {len(df.columns)} columnas")

bancos       ->     21 filas | 10 columnas
sensores     ->    100 filas | 10 columnas
personal     ->     62 filas | 8 columnas
lecturas     ->   2000 filas | 9 columnas


## 4. Lectura de la fuente API (`cortes_energia.json`)

Quinta fuente, de tipo distinto (JSON semiestructurado). ADF ya la copia
desde el endpoint REST simulado (DummyJSON) al Data Lake; acá se lee el
archivo aterrizado.

Primero se inspecciona la estructura real: si el JSON viene envuelto
(`{"cortes": [...]}`) hay que explotar el array antes de aplicar el esquema.

In [0]:
RUTA_CORTES = f"{RAW}/cortes_energia.json"

ESQUEMA_CORTES = StructType([
    StructField("corte_id",          StringType(), True),
    StructField("zona_electrica",    StringType(), True),
    StructField("fecha_hora_inicio", StringType(), True),
    StructField("fecha_hora_fin",    StringType(), True),
    StructField("duracion_min",      StringType(), True),
    StructField("causa",             StringType(), True),
])

# ADF escribió con File pattern "Set of objects" -> JSON Lines (un objeto por línea).
# multiLine debe ser false; con true, Spark parsea solo el primer objeto.
df_cortes = (spark.read
             .option("multiLine", "false")
             .schema(ESQUEMA_CORTES)
             .json(RUTA_CORTES))

df_cortes = (df_cortes
             .withColumn("archivo_origen", F.lit(RUTA_CORTES))
             .withColumn("fecha_carga",    F.current_timestamp()))

dfs["cortes_energia"] = df_cortes
df_cortes.createOrReplaceTempView("bronze_cortes_energia")

print(f"cortes_energia -> {df_cortes.count()} filas")   # esperado: 30
display(df_cortes)

cortes_energia -> 30 filas


corte_id,zona_electrica,fecha_hora_inicio,fecha_hora_fin,duracion_min,causa,archivo_origen,fecha_carga
CE-001,ZE-02,2025-06-05T10:30:00,2025-06-05T11:10:00,40,Mantenimiento programado,abfss://termored-central-us-dev@destorageintegration.dfs.core.windows.net/0_raw/cortes_energia.json,2026-08-12T18:30:22.6252Z
CE-002,ZE-03,2025-06-07T16:45:00,2025-06-07T20:45:00,240,Mantenimiento programado,abfss://termored-central-us-dev@destorageintegration.dfs.core.windows.net/0_raw/cortes_energia.json,2026-08-12T18:30:22.6252Z
CE-003,ZE-01,2025-06-02T08:15:00,2025-06-02T09:45:00,90,Tormenta eléctrica,abfss://termored-central-us-dev@destorageintegration.dfs.core.windows.net/0_raw/cortes_energia.json,2026-08-12T18:30:22.6252Z
CE-004,ZE-01,2025-06-06T13:30:00,2025-06-06T15:30:00,120,Sobrecarga de red,abfss://termored-central-us-dev@destorageintegration.dfs.core.windows.net/0_raw/cortes_energia.json,2026-08-12T18:30:22.6252Z
CE-005,ZE-05,2025-06-06T03:45:00,null,70,Mantenimiento programado,abfss://termored-central-us-dev@destorageintegration.dfs.core.windows.net/0_raw/cortes_energia.json,2026-08-12T18:30:22.6252Z
CE-006,ZE-01,2025-06-07T13:00:00,2025-06-07T17:00:00,240,Corte por obra en vía pública,abfss://termored-central-us-dev@destorageintegration.dfs.core.windows.net/0_raw/cortes_energia.json,2026-08-12T18:30:22.6252Z
CE-007,ZE-06,2025-06-07T23:15:00,2025-06-08T01:15:00,120,Sobrecarga de red,abfss://termored-central-us-dev@destorageintegration.dfs.core.windows.net/0_raw/cortes_energia.json,2026-08-12T18:30:22.6252Z
CE-008,ZE-01,2025-06-07T10:30:00,2025-06-07T11:10:00,40,Mantenimiento programado,abfss://termored-central-us-dev@destorageintegration.dfs.core.windows.net/0_raw/cortes_energia.json,2026-08-12T18:30:22.6252Z
CE-009,ZE-05,2025-06-04T21:45:00,2025-06-04T23:45:00,120,Sobrecarga de red,abfss://termored-central-us-dev@destorageintegration.dfs.core.windows.net/0_raw/cortes_energia.json,2026-08-12T18:30:22.6252Z
CE-010,ZE-06,2025-06-04T17:15:00,2025-06-04T18:25:00,9999,Sobrecarga de red,abfss://termored-central-us-dev@destorageintegration.dfs.core.windows.net/0_raw/cortes_energia.json,2026-08-12T18:30:22.6252Z


### 4.b (Alternativa) Consumo directo del endpoint REST

No es necesario si ADF ya aterrizó el archivo, pero sirve como respaldo si el endpoint de DummyJSON caduca.

In [0]:
# import requests, json
# URL = "https://dummyjson.com/c/c952-4cf3-46c4-ae80"
# payload = requests.get(URL, timeout=30).json()
# registros = payload if isinstance(payload, list) else payload.get("cortes", [])
# df_api = spark.createDataFrame(
#     [{k: (str(v) if v is not None else None) for k, v in r.items()} for r in registros]
# )
# display(df_api)

## 5. Inventario de la carga

Conteo de control antes de guardar en parquet.

In [0]:
inventario = [(n, df.count(), len(df.columns)) for n, df in dfs.items()]
display(spark.createDataFrame(inventario, ["fuente", "filas", "columnas"]))

fuente,filas,columnas
bancos,21,10
sensores,100,10
personal,62,8
lecturas,2000,9
cortes_energia,30,8


## 6. Persistencia en Bronze

In [0]:
for nombre, df in dfs.items():
    destino = f"{BRONZE}/{nombre}"
    ruta = guardar_parquet(df, destino, nombre)
    print(f"OK  {nombre:16s} -> {ruta}")

OK  bancos           -> abfss://termored-central-us-dev@destorageintegration.dfs.core.windows.net/1_bronze/bancos/bancos.parquet
OK  sensores         -> abfss://termored-central-us-dev@destorageintegration.dfs.core.windows.net/1_bronze/sensores/sensores.parquet
OK  personal         -> abfss://termored-central-us-dev@destorageintegration.dfs.core.windows.net/1_bronze/personal/personal.parquet
OK  lecturas         -> abfss://termored-central-us-dev@destorageintegration.dfs.core.windows.net/1_bronze/lecturas/lecturas.parquet
OK  cortes_energia   -> abfss://termored-central-us-dev@destorageintegration.dfs.core.windows.net/1_bronze/cortes_energia/cortes_energia.parquet


## 7. Verificación

Se relee lo escrito para confirmar que el archivo es legible y que el
recuento coincide con el origen.

In [0]:
for nombre in dfs:
    leido = spark.read.parquet(f"{BRONZE}/{nombre}")
    print(f"{nombre:16s} bronze={leido.count():6d}  origen={dfs[nombre].count():6d}")

display(spark.read.parquet(f"{BRONZE}/lecturas").limit(20))

bancos           bronze=    21  origen=    21
sensores         bronze=   100  origen=   100
personal         bronze=    62  origen=    62
lecturas         bronze=  2000  origen=  2000
cortes_energia   bronze=    30  origen=    30


lectura_id,sensor_id,timestamp,temperatura,humedad,bateria_pct,puerta_abierta,archivo_origen,fecha_carga
LEC-01861,SEN-0014,2025-06-08 23:01:00,-24.8,54.8,76.3,0,abfss://termored-central-us-dev@destorageintegration.dfs.core.windows.net/0_raw/lecturas.csv,2026-08-12T18:30:43.977214Z
LEC-00354,SEN-0009,2025-06-03 08:12:00,-12.2,39.0,94.1,0,abfss://termored-central-us-dev@destorageintegration.dfs.core.windows.net/0_raw/lecturas.csv,2026-08-12T18:30:43.977214Z
LEC-01334,SEN-0097,2025-06-06 23:06:00,4.7,47.6,67.2,0,abfss://termored-central-us-dev@destorageintegration.dfs.core.windows.net/0_raw/lecturas.csv,2026-08-12T18:30:43.977214Z
LEC-00906,SEN-0025,2025-06-05 08:10:00,22.7,50.7,66.5,0,abfss://termored-central-us-dev@destorageintegration.dfs.core.windows.net/0_raw/lecturas.csv,2026-08-12T18:30:43.977214Z
LEC-01290,SEN-0005,2025-06-06 16:13:00,22.2,43.7,89.4,0,abfss://termored-central-us-dev@destorageintegration.dfs.core.windows.net/0_raw/lecturas.csv,2026-08-12T18:30:43.977214Z
LEC-01274,SEN-0087,2025-06-06 16:11:00,4.2,59.3,62.9,0,abfss://termored-central-us-dev@destorageintegration.dfs.core.windows.net/0_raw/lecturas.csv,2026-08-12T18:30:43.977214Z
LEC-00939,SEN-0079,2025-06-05 16:01:00,-24.7,61.3,72.3,0,abfss://termored-central-us-dev@destorageintegration.dfs.core.windows.net/0_raw/lecturas.csv,2026-08-12T18:30:43.977214Z
LEC-01732,SEN-0025,2025-06-08 08:10:00,22.3,51.6,62.0,0,abfss://termored-central-us-dev@destorageintegration.dfs.core.windows.net/0_raw/lecturas.csv,2026-08-12T18:30:43.977214Z
LEC-00066,SEN-0092,2025-06-02 08:11:00,5.1,54.5,73.7,0,abfss://termored-central-us-dev@destorageintegration.dfs.core.windows.net/0_raw/lecturas.csv,2026-08-12T18:30:43.977214Z
LEC-01324,SEN-0019,06/06/2025 23:05,-24.1,48.2,94.5,0,abfss://termored-central-us-dev@destorageintegration.dfs.core.windows.net/0_raw/lecturas.csv,2026-08-12T18:30:43.977214Z
